# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LORA
* Model: answerdotai/ModernBERT-base
* Evaluation approach: accuracy
* Fine-tuning dataset: google-research-datasets/go_emotions

In [37]:
# https://medium.com/@mb20261/nlp-by-examples-text-classifications-with-transformers-fc2c14078208

import numpy as np
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}



## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [54]:
from datasets import load_dataset
# dataset = load_dataset("google-research-datasets/go_emotions") 
# dataset = load_dataset("imdb") 


dataset = load_dataset("financial_phrasebank", "sentences_allagree")
dataset = dataset["train"].train_test_split(test_size=0.1)
dataset["validation"] = dataset["test"]
del dataset["test"]

dataset


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2037
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 227
    })
})

In [ ]:
from transformers import (AutoTokenizer
                          , AutoModelForSequenceClassification
                          , BertForSequenceClassification
                          , Trainer
                          , TrainingArguments
)
from transformers import default_data_collator, get_linear_schedule_with_warmup
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

# model_name_or_path = 'bert-base-uncased'
# model_name_or_path = 'answerdotai/ModernBERT-base'
model_name_or_path = "bigscience/mt0-large"
model_name_or_path ='google-bert/bert-base-uncased'
# model_name_or_path ='google-bert/bert-large-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name_or_path
    # , num_labels=len(dataset["train"].features["label"].names)
    # ,id2label={0: "negative", 1: "neutral", 2: "positive"}
    # ,label2id={"negative": 0, "neutral": 1, "positive" : 2}
    )
dataset["train"].features["label"].names

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


['negative', 'neutral', 'positive']

In [57]:
model.classifier

Linear(in_features=768, out_features=2, bias=True)

In [80]:
from torch import nn 
model.classifier = nn.Linear(768, 3)
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_featu

In [81]:
from transformers import AdamW
from transformers import default_data_collator, get_linear_schedule_with_warmup

import torch 
from torch.utils.data import DataLoader
device='mps'
learning_rate = 2e-5
batch_size = 18
num_epochs = 6

In [102]:
# tokenizer(dataset['train']['text'], padding='max_length', truncation=True)
from transformers import default_data_collator, get_linear_schedule_with_warmup
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType
classes = dataset["train"].features["label"].names



def preprocess_function(examples):
    inputs = examples['sentence']
    targets = examples['label']
    model_inputs = tokenizer(
        inputs, 
        padding="max_length", 
        max_length=256, 
        truncation=True,
        return_tensors="pt"
    )
    model_inputs["labels"] = targets
    return model_inputs




In [92]:

preprocessed_datasets = dataset.map(preprocess_function, batched=True, batch_size=batch_size)

from torch.utils.data import default_collate
preprocessed_datasets['train'][0]
train_dataloader = DataLoader(preprocessed_datasets['train'], shuffle=True, collate_fn=default_data_collator
            ,batch_size=batch_size)
eval_dataloader = DataLoader(preprocessed_datasets['validation'], shuffle=True, collate_fn=default_data_collator
            ,batch_size=batch_size)


In [93]:
preprocessed_datasets['train']

print("Unique labels: ", len(set(preprocessed_datasets['train']['labels'])))
eval_dataloader.batch_size, train_dataloader.batch_size

Unique labels:  3


(18, 18)

In [94]:
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

# tokenizer_name_or_path = "bigscience/mt0-large"

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=32, lora_dropout=0.1
    , target_modules=['query','value']
    # ,modules_to_save=['classification_head']
)

lora_model = get_peft_model(model, peft_config)
# Load your LoRA weights
from torch import nn
# lora_model.classifier = nn.Linear(768, 3)  
lora_model.print_trainable_parameters()

trainable params: 1,181,955 || all params: 110,666,502 || trainable%: 1.0680


In [95]:
lora_model

model.parameters

<bound method Module.parameters of BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
         

In [96]:
optimizer = AdamW(lora_model.parameters(), lr=learning_rate)

In [97]:
# a = [x for x in train_dataloader]

# len(a[0]['labels'])


# len(batch['labels'])



In [98]:
lora_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default

In [99]:
# training and evaluation
from tqdm import tqdm
lora_model = lora_model.to(device)

for epoch in range(num_epochs):
    lora_model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        # print(batch)
        batch = {k: v.to(device) for k, v in batch.items()}
        # print("Length INput: ", len(batch['input_ids']))
        outputs = lora_model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    lora_model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = lora_model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            torch.argmax(outputs.logits, -1).detach().cpu().numpy()
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

  0%|          | 0/114 [00:00<?, ?it/s]


ValueError: Expected input batch_size (27) to match target batch_size (18).

In [100]:
File ~/Projects/Udacity GenAI/udacity_genai/.venv/lib/python3.12/site-packages/transformers/models/bert/modeling_bert.py:1064, in BertModel.forward(self, input_ids, attention_mask, token_type_ids, position_ids, head_mask, inputs_embeds, encoder_hidden_states, encoder_attention_mask, past_key_values, use_cache, output_attentions, output_hidden_states, return_dict)
from transformers.models.bert import modeling_bert

SyntaxError: invalid syntax (2320206774.py, line 1)

In [101]:
training_args = TrainingArguments(
    output_dir="bert_peft_trainer"
    , overwrite_output_dir = True
    , do_train=True, do_eval=True
    , eval_strategy = 'epoch'
    , logging_strategy = 'epoch'
    , per_device_eval_batch_size= batch_size
    , per_device_train_batch_size= batch_size
    , learning_rate = learning_rate 
    , optim='adamw_torch'
    , num_train_epochs = num_epochs
    )
bert_peft_trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=preprocessed_datasets['train'], # training dataset requires column input_ids
    eval_dataset=preprocessed_datasets['validation'],
    compute_metrics=compute_metrics,
)
bert_peft_trainer.train()

ValueError: Expected input batch_size (27) to match target batch_size (18).

In [162]:
bert_peft_trainer.model.save_pretrained("bert-peft")

In [147]:
lora_model.save_pretrained('./lora_pretrained')

In [163]:
lora_model.config._name_or_pathx

# Before training
print("Base model num_labels:", model.config.num_labels)
print("LoRA model num_labels:", lora_model.config.num_labels)

# # After loading the merged model
# print("Merged model num_labels:", merged_model.config.num_labels)

AttributeError: 'BertConfig' object has no attribute '_name_or_pathx'

In [164]:
from peft import PeftModel
original_model = AutoModelForSequenceClassification.from_pretrained(
  lora_model.config._name_or_path
)

original_with_adapter = PeftModel.from_pretrained(
  original_model, "bert-peft" # bert-peft; the folder of the saved adapter
)
merged_model = original_with_adapter.merge_and_unload()
merged_model.save_pretrained("merged-model")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [165]:
merged_model.to(device)
merged_model(**tokenized_val)

SequenceClassifierOutput(loss=None, logits=tensor([[-1.0716,  1.0649],
        [-0.9027,  0.9678],
        [-0.9796,  0.9547],
        [-0.7917,  0.8193],
        [-0.9592,  1.0755],
        [-0.1483, -0.0286],
        [-0.8373,  0.7191],
        [-0.8584,  0.9974],
        [-0.9792,  1.1208],
        [-0.2711, -0.0970],
        [-1.0292,  1.0461],
        [-1.0244,  0.9672],
        [-0.4921,  0.2198],
        [-0.4090, -0.0157],
        [-0.9025,  1.0728],
        [-0.8988,  1.0212],
        [-0.6711,  0.5679],
        [-1.0450,  1.2880],
        [-0.5318,  0.1824],
        [-0.8975,  1.0605],
        [-0.9369,  0.9721],
        [-1.0233,  1.1126],
        [-0.7729,  0.8245],
        [-1.0247,  1.1950],
        [-0.7897,  0.8876],
        [-1.0191,  1.0285],
        [-0.5525,  0.3465],
        [-0.9377,  1.0897],
        [-0.7915,  0.8292],
        [-0.9751,  0.9985],
        [-0.7875,  0.8705],
        [-0.2532, -0.0899],
        [-0.5944,  0.2649],
        [-0.5111,  0.3532],
     

In [25]:
torch.argmax(outputs.logits, axis=1), outputs.loss, eval_dataloader.dataset['label'][-batch_size:]
# torch.argmax(outputs.logits, -1).detach().cpu().numpy()
# torch.argmax(outputs.logits, -1).detach().cpu().numpy()

(tensor([0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0], device='mps:0'),
 tensor(0.0269, device='mps:0'),
 [1, 1, 1, 2, 1, 1, 0, 1, 1, 1, 1, 2])

### RETRY

In [148]:
from transformers import AutoModelForSequenceClassification
from peft import PeftModel
from transformers import (AutoTokenizer
                          , AutoModelForSequenceClassification
                          , BertForSequenceClassification
                          , Trainer
                          , TrainingArguments
)
from transformers import default_data_collator, get_linear_schedule_with_warmup
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType

In [ ]:

def preprocess_function(examples):
    inputs = examples['sentence']
    targets = examples['label']
    model_inputs = tokenizer(
        inputs, 
        padding="max_length", 
        max_length=256, 
        truncation=True,
        return_tensors="pt"
    )
    model_inputs["labels"] = targets
    return model_inputs




model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3  # specify number of labels
)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],  # fix the syntax here
)


learning_rate = 2e-5
batch_size = 8
num_epochs = 6

def preprocess_function(examples):
    inputs = examples['sentence']
    targets = examples['label']
    model_inputs = tokenizer(
        inputs, 
        padding="max_length", 
        max_length=256, 
        truncation=True
    )
    model_inputs["labels"] = targets
    return model_inputs

preprocessed_datasets = dataset.map(
    preprocess_function, 
    batched=True,
    remove_columns=dataset["train"].column_names  # Remove original columns
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,181,955 || all params: 110,666,502 || trainable%: 1.0680


Epoch,Training Loss,Validation Loss,Accuracy
1,1.012400,0.836042,0.687225
2,0.910400,0.791182,0.687225
3,0.844700,0.695975,0.696035
4,0.717200,0.575281,0.779736
5,0.635300,0.532546,0.784141
6,0.608500,0.522858,0.784141


TrainOutput(global_step=1530, training_loss=0.7880736008189083, metrics={'train_runtime': 576.2436, 'train_samples_per_second': 21.21, 'train_steps_per_second': 2.655, 'total_flos': 1630074927495168.0, 'train_loss': 0.7880736008189083, 'epoch': 6.0})

In [147]:

# Create the base model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# Training arguments
training_args = TrainingArguments(
    output_dir="default_bert_trainer",
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy='epoch',
    logging_strategy='epoch',
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    learning_rate=learning_rate,
    optim='adamw_torch',
    num_train_epochs= num_epochs
)

# Create and start trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=preprocessed_datasets['train'],
    eval_dataset=preprocessed_datasets['validation'],
    compute_metrics=compute_metrics,
)
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.600600,0.508723,0.779736
2,0.594000,0.503742,0.779736
3,0.591900,0.502092,0.784141
4,0.590000,0.501284,0.792952
5,0.589700,0.500866,0.792952
6,0.585800,0.500710,0.792952


TrainOutput(global_step=1530, training_loss=0.5919771580914267, metrics={'train_runtime': 353.2897, 'train_samples_per_second': 34.595, 'train_steps_per_second': 4.331, 'total_flos': 1630074927495168.0, 'train_loss': 0.5919771580914267, 'epoch': 6.0})

In [152]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [161]:
trainer.save_model('default_bert_model')
pre_model = AutoModelForSequenceClassification.from_pretrained('default_bert_model', num_labels=3)
pre_model.to(device)

t = tokenizer(dataset['validation']['sentence'], padding=True, truncation=True,return_tensors='pt')
t.to(device)
outputs = pre_model(**t)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [169]:
compute_metrics((outputs.logits.cpu().detach().numpy(),dataset['validation']['label'] ))



{'accuracy': np.float64(0.7929515418502202)}

In [ ]:

# Create LoRA config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

# Create LoRA model
lora_model = get_peft_model(model, peft_config)
lora_model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="bert_peft_trainer",
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy='epoch',
    logging_strategy='epoch',
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    learning_rate=learning_rate,
    optim='adamw_torch',
    num_train_epochs= num_epochs
)

# Create and start trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=preprocessed_datasets['train'],
    eval_dataset=preprocessed_datasets['validation'],
    compute_metrics=compute_metrics,
)

trainer.train()

In [145]:
trainer.save_model("bert_classifier_final")
from peft import PeftModelForSequenceClassification
loaded_model = PeftModelForSequenceClassification.from_pretrained(model, "bert_classifier_final")

In [140]:
outputs = model(**validation_tokenized)

In [125]:
from peft import PeftModel
validation_tokenized = tokenizer(dataset['validation']['sentence'], padding=True, truncation=True, return_tensors='pt')
validation_tokenized.to(device)
print(validation_tokenized)
outputs = lora_model(**validation_tokenized)


{'input_ids': tensor([[ 101, 1996, 3643,  ...,    0,    0,    0],
        [ 101, 1996, 2034,  ...,    0,    0,    0],
        [ 101, 1996, 2707,  ...,    0,    0,    0],
        ...,
        [ 101, 6983, 2924,  ...,    0,    0,    0],
        [ 101, 1996, 3155,  ...,    0,    0,    0],
        [ 101, 1996, 3930,  ...,    0,    0,    0]], device='mps:0'), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='mps:0')}


In [ ]:
# torch.argmax(outputs.logits, axis=1)
# dataset['validation']['label']



{'accuracy': np.float64(0.7841409691629956)}

array([[-1.084637  ,  1.225609  , -0.23401234],
       [-1.3039601 ,  1.5671574 , -0.33476457],
       [-1.5006039 ,  1.7496247 , -0.34419644],
       [-1.7290581 ,  2.198359  , -0.47018084],
       [-1.3103527 ,  1.5316836 , -0.2397772 ],
       [-1.7601162 ,  2.0357747 , -0.47549322],
       [-1.7846684 ,  2.1045642 , -0.44123933],
       [ 0.13353984, -0.9340858 ,  0.56161475],
       [-1.4412314 ,  1.6570082 , -0.25146088],
       [-1.4308376 ,  1.6029435 , -0.23897733],
       [ 0.3216268 , -0.57214457,  0.45171106],
       [-1.5779785 ,  1.8594652 , -0.43725443],
       [-1.7594473 ,  2.0528474 , -0.43187714],
       [-1.5991194 ,  1.7676309 , -0.4362293 ],
       [-1.3235259 ,  1.7682623 , -0.29320532],
       [-1.4991641 ,  1.7631754 , -0.34189615],
       [-1.2857355 ,  1.5507592 , -0.22653385],
       [-1.4667878 ,  1.749101  , -0.3185767 ],
       [-1.7712188 ,  1.9505816 , -0.45475614],
       [-0.4659963 ,  0.56659603,  0.11594266],
       [-1.0026724 ,  1.2378708 , -0.176

In [117]:
lora_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default

In [58]:
# print accuracy
correct = 0
total = 0
for pred, true in zip(eval_preds, dataset["validation"]['label']):
    if pred == true:
        correct += 1
    total += 1
accuracy = correct / total * 100
print(f"{accuracy=} % on the evaluation dataset")

accuracy=45.81497797356828 % on the evaluation dataset


In [60]:

reloaded_lora = original_with_adapter = PeftModel.from_pretrained(
  original_model, "bert-peft" # bert-peft; the folder of the saved adapter
)

reloaded_lora = reloaded_lora.to(device)

In [ ]:
with torch.no_grad():
    reloaded_lora(**)

In [ ]:
from peft import AutoPeftModelForSequenceClassification, AutoPeftModelForSeq2SeqLM
device='mps'

reloaded_lora = AutoPeftModelForSequenceClassification.from_pretrained('./lora_pretrained'
                                                                       , is_trainable=False
                                                                       ,num_labels=len(dataset["train"].features["label"].names)
                                                                       )
reloaded_lora = reloaded_lora.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: Error(s) in loading state_dict for PeftModelForSequenceClassification:
	size mismatch for base_model.model.classifier.modules_to_save.default.weight: copying a param with shape torch.Size([2, 768]) from checkpoint, the shape in current model is torch.Size([3, 768]).
	size mismatch for base_model.model.classifier.modules_to_save.default.bias: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([3]).

In [ ]:
lora_model.config.num_lae

In [63]:
from datasets import Dataset
few_shot_examples = {"sentence" : ['the bull market is starting to look like a bear',
                        'Stocks went in the direction of -100bps',
                        'we can see nvidia is gaining steam event with the government cracking down',
                        'wow, buy this stock now - its gone up 500% and continuing to grow',

                        ],
         "label" : [0,0,2,2]
        }

fse = Dataset.from_dict(few_shot_examples)
new_examples= fse.map(preprocess_function, batched=True, remove_columns=['sentence','label'], keep_in_memory=False)
new_examples = new_examples.with_format("torch", device=device)
# new_examples.to(device)

new_examples

Map: 100%|██████████| 4/4 [00:00<00:00, 1489.19 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [64]:
fse['sentence']

['the bull market is starting to look like a bear',
 'Stocks went in the direction of -100bps',
 'we can see nvidia is gaining steam event with the government cracking down',
 'wow, buy this stock now - its gone up 500% and continuing to grow']

In [65]:
fse = fse.with_format("torch", device=device)
ds = tokenizer(fse['sentence'], max_length=512, padding=True , truncation=True, return_tensors='pt')
ds = ds.to(device)
# ds = Dataset.from_dict(tokenizer(fse['sentence'])).with_format("torch", device=device)


AttributeError: 'BertForSequenceClassification' object has no attribute 'batch'

In [115]:
# tokenizer(dataset['validation']
tokenized_val = tokenizer(dataset['validation']['sentence'], return_tensors='pt', padding=True, truncation=True)        
tokenized_val.to(device)  
new_outputs = reloaded_lora(**tokenized_val)

new_outputs.logits.argmax(axis=1)


tensor([1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1,
        0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1,
        1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1,
        1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
        0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
        0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1,
        1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0], device='mps:0')

In [104]:
reloaded_lora.eval()

# tokenizer(fse)
with torch.no_grad():
        new_outputs = reloaded_lora(**preprocessed_datasets['validation'])

with torch.no_grad():
        old_model_outputs = model(**preprocessed_datasets['validation'])

new_outputs, old_model_outputs

TypeError: PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                   

In [ ]:
a = preprocessed_datasets['validation']



{'sentence': ['The first stage of the contract covers 133 stores and 600 cash registers .',
  'Unbelievably , the company that makes them - Fiskars Corporation - was formed in 1649 when a Dutch merchant named Peter Thorwoste was given a charter to establish a blast furnace and forging operation in the small Finnish village of Fiskars .',
  "One price category is for calls on the preferred operator 's network , and another for calls on other operators ' networks .",
  "`` This is a significant milestone for Benefon , helping us to secure critical USP 's for our personal navigation product roadmap for 2007 and beyond , '' commented Simon Button , Chief Technology Officer at Benefon .",
  "Maggie Ramsey 's wait - and those of thousands of Oregon and Washington guides , anglers and others who flock to his frequent seminars - is nearly over .",
  "In the second quarter of 2010 , Raute 's net loss narrowed to EUR 123,000 from EUR 1.5 million in the same period of 2009 .",
  'As an alternativ

In [82]:

# tokenizer(fse)
with torch.no_grad():
        new_outputs = reloaded_lora(**)

with torch.no_grad():
        old_model_outputs = model(**ds)

new_outputs, old_model_outputs

TypeError: PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                   

In [71]:
torch.argmax(new_outputs.logits, dim=-1), torch.argmax(old_model_outputs.logits, dim=-1)


(tensor([1, 1, 0, 1], device='mps:0'), tensor([1, 1, 0, 1], device='mps:0'))

In [68]:
new_outputs.logits.exp()

tensor([[1.6629e-02, 3.7769e+01],
        [4.5750e-03, 1.0514e+02],
        [2.7091e+01, 6.0916e-02],
        [8.1168e-03, 5.1564e+01]], device='mps:0')

In [ ]:
torch.argmax(new_outputs.logits, dim=-1)

tensor([1, 1, 1], device='mps:0')

In [96]:
# tokenized_dataset['train']['input_ids']


training_args = TrainingArguments(
    output_dir='./peft_train',          # Output directory
    eval_strategy="epoch",     # Evaluation strategy to adopt during training
    learning_rate=learning_rate,              # Learning rate for training
    per_device_train_batch_size=batch_size,   # Batch size for training
    per_device_eval_batch_size=batch_size,    # Batch size for evaluation
    num_train_epochs=2,               # Total number of training epochs
    weight_decay=0.20,                # Strength of weight decay
    use_mps_device  = True
)


In [230]:
trainer = Trainer(
    model=lora_model,                         # The instantiated 🤗 Transformers model to be trained
    args=training_args,                  # Training arguments, defined above
    train_dataset=train_dataloader.dataset,
    eval_dataset=eval_dataloader.dataset, # Evaluation dataset
    # compute_metrics=compute_metrics
)


In [231]:
trainer.train()


Epoch,Training Loss,Validation Loss


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (2, 227) + inhomogeneous part.

In [25]:


optimizer = AdamW(model.parameters(), lr = learning_rate)


device = torch.device("cuda" if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else  "cpu")
# print("Device: %s" % device)
# model.to(device)

train_dataloader = DataLoader()

import tqdm
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm.tqdm(dataset['train'])):
        tokenized_batch = tokenizer(batch['text'], padding="max_length", truncation=True, return_tensors='pt')
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        # lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm.tqdm(dataset['validation'])):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

/Users/danielfrimer/Projects/Udacity GenAI/udacity_genai/.venv/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/43410 [00:00<?, ?it/s]

  0%|          | 0/43410 [00:00<?, ?it/s]


AttributeError: 'str' object has no attribute 'to'

In [41]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e